# notebook for developing components of the crm datalake ingestion method

In [ ]:
#import statements
import pandas as pd
import os, sys, json, gc, sqlalchemy, re, csv, math
from datetime import datetime
import TELTools as TT
from concurrent.futures import ThreadPoolExecutor
import threading
from azure.storage.blob import BlobClient, ContainerClient
import io, gzip


In [ ]:
# globals
batch_datetimestamp = datetime.now()
CURRENT_TIMESTAMP = str(datetime.now()).replace(" ","_").replace(":","_").replace(".","_")
tel_tools_config_path = 'C:\\Sources\\Repos\\EnterpriseCloud\\Configuration\\TELTools.json'
ENV = os.environ['SNOWFLAKE_ENV']
blob_folder = '/CRM-' + CURRENT_TIMESTAMP + '-' + ENV + '/'

dtype_conversions = {
    'boolean': 'BOOLEAN', 
    'datetime': 'TIMESTAMP_LTZ', 
    'decimal': 'NUMBER(38,10)', 
    'double': 'NUMBER(38,10)', 
    'guid': 'VARCHAR(16777276)', 
    'int64': 'NUMBER(38,0)', 
    'string': 'VARCHAR'
}

default_file_format = {
    'columnHeaders': False,
    'delimiter': ',',
    'quoteStyle': 'QuoteStyle.Csv',
    'csvStyle': 'CsvStyle.QuoteAlways',
    'encoding': 'UTF-8'
}

quote_styles = {
    'CsvStyle.QuoteAlways': csv.QUOTE_ALL, 
    'CsvStyle.QuoteAfterDelimiter': csv.QUOTE_ALL # will need to look deeper into potential quote styles
}

all_header_formats = {}


for i in range(math.ceil(math.log10(sys.maxsize))):
    try:
        csv.field_size_limit(int(sys.maxsize/(10**i)))
        break
    except:
        continue


In [ ]:
# for local testing in DEV

import snowflake.connector as sc
from snowflake.connector.pandas_tools import write_pandas

def putSnowflakeDataFrame_JBS(db,schema,table_name,json_list,fileName,fileDate,row_count):
    messages = []
    conn = sc.connect(
        account = 'telus-global.privatelink',
        user = 'jay.spiers@us.tel.com',
        authenticator = 'externalbrowser',
        database = 'LZ_DEV',
        schema = 'CRM_JBS',
        warehouse = 'DEVELOPER_WH',
        role = 'SYSADMIN_DEV'
    )

    # messages.append(message_conn)
    if conn != 'no connection':
        #create a new dataframe the size of the incoming data with LZ schema columns.
        # set data objects to strings
        uploadDate = str(datetime.now())
        fileDate =str(fileDate)

        # create a data frame with control columns of correct size
        columns = ['_FILENAME','_FILEDATE','_UPLOADDATE']
        df_write = pd.DataFrame(columns=columns)
        data = [fileName,fileDate,uploadDate]
        df_write.loc[0] = data
        df_new = pd.concat([df_write]*row_count,ignore_index=True)
        # add the incoming sapapi data
        df_new['JSONFIELD'] = json_list
        
        # write to sf: https://docs.snowflake.com/developer-guide/python-connector/python-connector-api#write_pandas
        message_write = write_pandas(conn,df_new,table_name,db,schema)
        messages.append(f'putSnowflakeDataFrame(): Insert into snowflake completed for {db}, {schema}, {table_name}, {fileName} - {message_write}')


In [ ]:
# functions
def getBlobVars(blob_key):
    global blob_url_root
    blob_url_root = TT.getAzureSecret(blob_key).value # this needs its own key, remove the replace

# load config entries from TELTools.json
def getTELToolsConfig():
    with open(tel_tools_config_path,'r') as f:
        tel_tools_json = json.load(f,strict=False)
        crm_json = tel_tools_json.get('Services').get('CRM').get(ENV)
        FILEUPLOADER_CONFIG = crm_json.get('FILEUPLOADER_CONFIG')
        THREADS = int(crm_json.get('THREADS'))
        # get last incremental value (date from last run) - store in file in runtime config folder 
        RUN_FILE = crm_json.get('RUN_FILE')
    return FILEUPLOADER_CONFIG, THREADS, RUN_FILE

# get incremental value
def getLastRunData(file_path):
    LastRunData = None
    last_run_success = False
    try: 
        if os.path.isfile(file_path):
            with open(file_path,'r',encoding='utf-8-sig') as f:
                last_value_json = json.load(f,strict=False)
                LastRunData = last_value_json.get('LastRunData')
            message = f'Sucessfully opened incremental file {file_path} and loaded {LastRunData}.'
            last_run_success = True
            return last_run_success, message, LastRunData
    except Exception as e:
        message = f'Failed to open {file_path}: {e}.'
        return last_run_success, message, LastRunData


def processEntity(vars):

    entity_name, run_name, src_container, tgt_container, model_json, file_uploader_config, partition_file_formats = vars
    
    messages = []
    failure_count = 0

    file_config_dict = file_uploader_config[entity_name.upper()]

    try: # use the TrickleFeedService when applicable, need to confirm if this is the standard
        cdm_path = f'{run_name}/Microsoft.Athena.TrickleFeedService/{entity_name}/{entity_name}.cdm.json'
        cdm_blob = src_container.download_blob(cdm_path)
        cdm_json = json.loads(cdm_blob.readall().decode('utf-8-sig'))
        attributes = cdm_json['definitions'][0]['hasAttributes']
        header_formats = {column['name']: column['dataFormat'].lower() for column in attributes}
    except: # when TrickleFeedService fails, use the model.json
        attributes = model_json['entities'][0]['attributes']
        header_formats = {a['name']: a['dataType'].lower() for a in attributes}
    # need to wrap this an another try catch and use default header_formats if needed from tehsnowflakesa

    all_header_formats[entity_name] = header_formats

    entity_path = f'{run_name}/{entity_name}/'

    total_row_count = 0
    for blob in src_container.walk_blobs(name_starts_with=entity_path):

        print('\n'+str(datetime.now()), blob.name)
        partition_file_name = blob.name.split('/')[-1]

        try: names = list(header_formats.keys())
        except: names = []

        try: file_format = partition_file_formats[entity_name][partition_file_name[:-4]]
        except: file_format = default_file_format

        try: sep = file_format['delimiter'] 
        except: sep = ','

        try: encoding = file_format['encoding'] 
        except: encoding = 'UTF-8'

        try: quoting = quote_styles[file_format['csvStyle']] 
        except: quoting = csv.QUOTE_ALL

        blob_content = src_container.download_blob(blob.name).readall()
        blob_decompress = blob_content.decode('utf-8-sig')
        blob_string = io.StringIO(blob_decompress)

        chunk_count = 0
        
        for file_chunk in pd.read_csv(blob_string, names=names, sep=sep, chunksize=100_000, engine="python", encoding=encoding, quoting=quoting):
            chunk_count += 1
            row_count = len(file_chunk.index)
            total_row_count += row_count
        
            # process to json and post chunk to snowflake
            if row_count > 0:
                try:
                    db = file_config_dict['SNOWFLAKE_DATABASE_NAME']
                    schema = file_config_dict['SNOWFLAKE_SCHEMA_NAME']
                    table_name = file_config_dict['SNOWFLAKE_TABLE_NAME']
                    new_file_name_txt = f'{run_name}-{entity_name}-{partition_file_name}-{chunk_count}.json'
                    filedate = datetime.strptime(run_name, '%Y-%m-%dT%H.%M.%SZ')
                    json_list = file_chunk.apply(lambda x: json.loads(x.to_json()), axis=1).to_list()
                    
                    message = putSnowflakeDataFrame_JBS(db,schema,table_name,json_list,new_file_name_txt,filedate,row_count) # refactor to use main function
                    message = f'processEntity() for chunk {chunk_count} and {row_count} rows: {message}'
                    print(datetime.now(), message)
                    messages.append(message)
                    if 'FAIL_SIGNAL' in message: failure_count += 1

                except Exception as e:
                    db = file_config_dict['SNOWFLAKE_DATABASE_NAME']
                    schema = file_config_dict['SNOWFLAKE_SCHEMA_NAME']
                    table_name = file_config_dict['SNOWFLAKE_TABLE_NAME']
                    new_file_name_txt = f'{run_name}-{entity_name}-{partition_file_name}-{chunk_count}.json'
                    message = f"Error writing to snowflake: {db}, {schema}, {table_name}, {new_file_name_txt}: {e}"
                    print(datetime.now(), message)
                    messages.append(message)
                    failure_count += 1
                
        tgt_container.upload_blob('ARCHIVE/'+blob.name, blob_content, overwrite=True)
    
    print()
    processMetadata(entity_name, run_name, src_container, tgt_container, file_uploader_config, messages, failure_count)
    
    final_message = f'{datetime.now()} Succesfully processed {entity_name} and uploaded a total of {row_count} rows with a total of {failure_count} errors'
    messages.append(final_message)
    print(final_message)


    return messages


def processMetadata(entity_name, run_name, src_container, tgt_container, file_uploader_config, messages, failure_count):
    
    metadata_path = f'{run_name}/Microsoft.Athena.TrickleFeedService/{entity_name}-EntityMetadata.json'
    print(f'\n{dateteim.now()} {metadata_path}')

    blob_content = src_container.download_blob(metadata_path).readall()
    blob_decompress = blob_content.decode('utf-8-sig')
    blob_json = json.loads(blob_decompress)

    metadata_entities = ['StateMetadata', 'StatusMetadata', 'GlobalOptionSetMetadata', 'OptionSetMetadata', 'TargetMetadata', 'AttributeMetadata']
    for meta_name in metadata_entities:
        df = pd.DataFrame(blob_json[meta_name], dtype='string')

        file_config_dict = file_uploader_config[meta_name.upper()]

        db = file_config_dict['SNOWFLAKE_DATABASE_NAME']
        schema = file_config_dict['SNOWFLAKE_SCHEMA_NAME']
        table_name = file_config_dict['SNOWFLAKE_TABLE_NAME']
        new_file_name_txt = f'{run_name}-{entity_name}-{meta_name}.json'
        filedate = datetime.strptime(run_name, '%Y-%m-%dT%H.%M.%SZ')
        row_count = len(df)
        json_list = df.apply(lambda x: json.loads(x.to_json()), axis=1).to_list()

        message = putSnowflakeDataFrame_JBS(db,schema,table_name,json_list,new_file_name_txt,filedate,row_count)
        message = f'processMetadata() for table {meta_name} and {row_count} rows: {message}'
        print(datetime.now(), message)
        messages.append(message)
        if 'FAIL_SIGNAL' in message: failure_count += 1

    tgt_container.upload_blob(f'ARCHIVE/{run_name}/{entity_name}/EntityMetadata.json', blob_content, overwrite=True)



In [ ]:
# get tel tools config - add as a service to TELTools.json
FILEUPLOADER_CONFIG, THREADS, RUN_FILE = getTELToolsConfig()
FILEUPLOADER_CONFIG, THREADS, RUN_FILE

In [ ]:
# get file uploader config - copy of existing file uploader for crm
file_uploader_config = TT.getFileUploaderConfig(FILEUPLOADER_CONFIG)
file_uploader_config

In [ ]:
sas_key_src = 'AZURE-STORAGE-CRM-SAS-PROD1'
sas_url_src = TT.getAzureSecret(sas_key_src).value
src_container = ContainerClient.from_container_url(sas_url_src)

sas_key_tgt = 'AZURE-STORAGE-CRM-SAS-SFSA'
sas_url_tgt = TT.getAzureSecret(sas_key_tgt).value
tgt_container = ContainerClient.from_container_url(sas_url_tgt)


In [ ]:
blob = srcContainer.download_blob('2023-07-11T08.42.03Z/model.json')
incident = json.loads(blob.readall().decode('utf-8-sig'))

blob = srcContainer.download_blob('2023-05-18T02.57.02Z/model.json')
opportunity = json.loads(blob.readall().decode('utf-8-sig'))

blob = srcContainer.download_blob('2023-05-13T01.42.02Z/model.json')
tel_sm_bulk_fcst = json.loads(blob.readall().decode('utf-8-sig'))

blob = srcContainer.download_blob('2023-05-18T01.57.03Z/model.json')
tel_sm_sirfisfcst = json.loads(blob.readall().decode('utf-8-sig'))

dtype_conversions = {
    'boolean': 'BOOLEAN', 
    'datetime': 'TIMESTAMP_LTZ(9)', 
    'datetimeoffset': 'TIMESTAMP_LTZ(9)', 
    'decimal': 'NUMBER(38,10)', 
    'double': 'NUMBER(38,10)', 
    'guid': 'VARCHAR(16777276)', 
    'int64': 'NUMBER(38,0)', 
    'string': 'VARCHAR(16777276)'
}

tables, columns, dtypes = ([], [], [])
for entity in incident['entities']:
    if entity['name']=='incident':
        new_columns = [a['name'] for a in entity['attributes']]
        tables.extend([entity['name']]*len(new_columns))
        columns.extend(new_columns)
        dtypes.extend([a['dataType'] for a in entity['attributes']])
for entity in opportunity['entities']:
    if entity['name']=='opportunity':
        new_columns = [a['name'] for a in entity['attributes']]
        tables.extend([entity['name']]*len(new_columns))
        columns.extend(new_columns)
        dtypes.extend([a['dataType'] for a in entity['attributes']])
for entity in tel_sm_bulk_fcst['entities']:
    if entity['name']=='tel_sm_bulk_fcst':
        new_columns = [a['name'] for a in entity['attributes']]
        tables.extend([entity['name']]*len(new_columns))
        columns.extend(new_columns)
        dtypes.extend([a['dataType'] for a in entity['attributes']])
for entity in tel_sm_sirfisfcst['entities']:
    if entity['name']=='tel_sm_sirfisfcst':
        new_columns = [a['name'] for a in entity['attributes']]
        tables.extend([entity['name']]*len(new_columns))
        columns.extend(new_columns)
        dtypes.extend([a['dataType'] for a in entity['attributes']])

df = pd.DataFrame({'Table': tables, 'Column': columns, 'CRM Type': dtypes})

blob = srcContainer.download_blob('2023-07-01T00.12.03Z/Microsoft.Athena.TrickleFeedService/msdyn_workorder-EntityMetadata.json')
meta_json = json.loads(blob.readall().decode('utf-8-sig'))
meta_schema = [[k, key, 'string'] for k, v in meta_json.items() for key in v[0].keys()]
meta_df = pd.DataFrame(meta_schema, columns=['Table', 'Column', 'CRM Type'])
df = df.append(meta_df, ignore_index=True)

df['SF Type'] = df['CRM Type'].str.lower().map(dtype_conversions)
df.to_csv('missing_schema.csv', index=False)
df


In [ ]:
last_run_success, message, LastRunData = getLastRunData(RUN_FILE)

changelog_path = 'Changelog/changelog.info'
changelog_blob = src_container.download_blob(changelog_path)
newest_run = changelog_blob.readall().decode('utf-8-sig')
LastRunData, newest_run

In [ ]:
pattern = '\\d{4}-\\d{2}-\\d{2}T\\d{2}.\\d{2}.\\d{2}Z/'
run_names = []

for blob in src_container.walk_blobs():
    if re.match(pattern, blob.name):
        run_name = blob.name.split('/')[-2]
        if run_name > LastRunData and run_name <= newest_run:
            run_names.append(run_name)
run_names

In [ ]:
all_header_formats = {} # convert this to read current version in snowflakesa

messages = []

# run_name = run_names[0]
for run_name in run_names:
    model_blob = src_container.download_blob(run_name+'/model.json')
    model_json = json.loads(model_blob.readall().decode('utf-8-sig'))

    partition_file_formats = {e['name']:{p['name']:p['fileFormatSettings'] if e['partitions'] else default_file_format for p in e['partitions']} for e in model_json['entities']}

    ignores = ['Microsoft.Athena.TrickleFeedService', 'OptionsetMetadata']

    args = []
    for blob in src_container.walk_blobs(name_starts_with=run_name+'/'):
        entity_name = blob.name.split('/')[-2]
        if blob.name.endswith('/') and entity_name not in ignores:
            args.append([entity_name, run_name, src_container, tgt_container, model_json, file_uploader_config, partition_file_formats])
            # print(datetime.now(), entity_name, run_name)
            # message = processEntity(args[0])
            # messages.append(message)
            # break

    print(f'\n\n{datetime.now()} Run {run_name} found {len(args)} entities from {args[0][0].upper()} to {args[-1][0].upper()}\n')

    # thread process args
    with ThreadPoolExecutor(max_workers=THREADS) as exe:
        print(datetime.now(), f'Threading max workers: {exe._max_workers}.')
        # Maps the method with a list of values.
        results = exe.map(processEntity, args)

    # save header formats to telsnowflakesa
    all_header_formats_txt = json.dumps(all_header_formats)
    tgt_container.upload_blob('headers.json', all_header_formats_txt, overwrite=True)
    
    # update runtime config last run value
    last_value_json = {"LastRunData": run_name}
    with open(RUN_FILE,'w',encoding='utf-8-sig') as f:
        f.write(json.dumps(last_value_json))
    
    # break

messages

#### Update crm.FILE_EXTRACTS.json

In [ ]:
import snowflake.connector as sc
from concurrent.futures import ThreadPoolExecutor
import json
import TELTools as TT
from azure.storage.blob import BlobClient, ContainerClient


conn = sc.connect(
                account = 'telus-global.privatelink'
                ,user = 'jay.spiers@us.tel.com'
                ,authenticator = 'externalbrowser'
                ,database = 'LZ_PROD'
                ,schema = 'CRM_FILE'
                ,warehouse = 'DEVELOPER_WH'
                ,role = 'SYSADMIN_DEV')

curs = conn.cursor()

tables = sorted(curs.execute("select distinct TABLE_NAME from INFORMATION_SCHEMA.TABLES where TABLE_SCHEMA = 'CRM_FILE'").fetch_pandas_all()['TABLE_NAME'])

run_data = {"LastDefaultRun": '', "RunData": {}}

def update_run_data(table):
    query = f"select distinct left(_FILENAME, 20) FILE from {table}"
    res = curs.execute(query).fetch_pandas_all()
    fnames_sorted = sorted(res['FILE'])
    previous_run = fnames_sorted[-1]
    run_data['RunData'][table.lower()] = fnames_sorted
    if previous_run > run_data['LastDefaultRun']:
        run_data['LastDefaultRun'] = previous_run

with ThreadPoolExecutor(max_workers=16) as exe:
    exe.map(update_run_data, tables)

run_data['RunData'] = {k:run_data['RunData'][k] for k in sorted(run_data['RunData'])}

with open('F:\\Shares\\Files\\TokyoElectron\\Python\\RuntimeConfig\\crm.FILE_EXTRACTS.json', 'w') as f:
    f.write(json.dumps(run_data, indent=4))
with open('S:\\Files\\TokyoElectron\\Python\\RuntimeConfig\\crm.FILE_EXTRACTS.json', 'w') as f:
    f.write(json.dumps(run_data, indent=4))
# with open('Q:\\Files\\TokyoElectron\\Python\\RuntimeConfig\\crm.FILE_EXTRACTS.json', 'w') as f:
#     f.write(json.dumps(run_data, indent=4))

In [ ]:
sas_key_tgt = 'AZURE-STORAGE-CRM-SAS-SFSA'
sas_url_tgt = TT.getAzureSecret(sas_key_tgt).value
tgt_container = ContainerClient.from_container_url(sas_url_tgt)

def move_blob(blob_name):
    if not blob_name.endswith('/'):
        run_match = re.search('^MANIFEST/(\\d{4}-\\d{2}-\\d{2}T\\d{2}.\\d{2}.\\d{2}Z).txt$', blob_name)
        entity_match = re.search('^MANIFEST/(\\d{4}-\\d{2}-\\d{2}T\\d{2}.\\d{2}.\\d{2}Z)-(\\w+).txt$', blob_name)

        if run_match:
            r = run_match.groups()[0]
            new_path = f'MANIFEST/{r}/{r}.txt'
            blob_content = tgt_container.download_blob(blob_name).readall()
            tgt_container.upload_blob(new_path, blob_content, overwrite=True)
            tgt_container.delete_blob(blob_name)
            print(f'moved {blob_name} to {new_path}')
        elif entity_match:
            r, e = entity_match.groups()
            new_path = f'MANIFEST/{r}/{e}.txt'
            blob_content = tgt_container.download_blob(blob_name).readall()
            tgt_container.upload_blob(new_path, blob_content, overwrite=True)
            tgt_container.delete_blob(blob_name)
            print(f'moved {blob_name} to {new_path}')


blob_list = tgt_container.walk_blobs(name_starts_with='MANIFEST/')
with ThreadPoolExecutor(max_workers=32) as exe:
    exe.map(move_blob, blob_list)

In [ ]:
tables

In [ ]:
sas_key_tgt = 'AZURE-STORAGE-CRM-SAS-SFSA'
sas_url_tgt = TT.getAzureSecret(sas_key_tgt).value
tgt_container = ContainerClient.from_container_url(sas_url_tgt)

In [ ]:
tgt_container.download_blob('MANIFEST/account-bkp.txt').

In [ ]:
empty_tables = [k for k, v in run_data['RunData'].items() if not v]
query = f"select TABLE_NAME, ROW_COUNT from LZ_DEV.INFORMATION_SCHEMA.TABLES where TABLE_SCHEMA = 'CRM_FILE' and TABLE_NAME in {str(tuple(empty_tables)).upper()} and ROW_COUNT > 0"
false_empty_tables = sorted(conn.cursor().execute(query).fetch_pandas_all()['TABLE_NAME'])
false_empty_tables